# 06 — Logistic Regression

Purpose: strong, interpretable linear baseline with class balancing and grouped hyperparameter tuning.

In [1]:
# AI-Based IAM Permission Optimizer — Model V2
# Run notebooks in order: 01 → 10
# Raw CSVs should be available in the project root or adjust RAW_DIR below.
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.metrics import average_precision_score
from scipy.stats import randint, loguniform

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_DIR / "labeled_dataset.csv")
splits = pd.read_csv(DATA_DIR / "splits.csv")
df["split"] = splits["split"].values

# Model deliberately excludes identifiers and exact action/resource strings.
candidate_numeric = [
    "usage_count", "log_usage_count", "unique_days_used",
    "log_unique_days_used", "days_since_last_use", "active_days",
    "usage_frequency", "success_rate", "smoothed_failure_rate",
    "risk_weight", "scope_breadth", "is_wildcard_resource",
    "is_delete", "is_permission_change", "is_security_sensitive"
]
candidate_categorical = [
    "service", "operation_type", "risk_level", "action_family",
    "position", "department", "team"
]

numeric_features = [c for c in candidate_numeric if c in df.columns]
categorical_features = [c for c in candidate_categorical if c in df.columns]

all_features = numeric_features + categorical_features
constant = [c for c in all_features if df[c].nunique(dropna=False) <= 1]
numeric_features = [c for c in numeric_features if c not in constant]
categorical_features = [c for c in categorical_features if c not in constant]
all_features = numeric_features + categorical_features

X = df[all_features].copy()
y = df["target"].copy()
groups = df["user_id"].copy()

fit_mask = df["split"].eq("train")
val_mask = df["split"].eq("validation")
test_mask = df["split"].eq("test")

X_fit, y_fit, groups_fit = X.loc[fit_mask], y.loc[fit_mask], groups.loc[fit_mask]
X_val, y_val, groups_val = X.loc[val_mask], y.loc[val_mask], groups.loc[val_mask]
X_test, y_test, groups_test = X.loc[test_mask], y.loc[test_mask], groups.loc[test_mask]

print("Features:", all_features)
print("Rows -> train:", len(X_fit), "validation:", len(X_val), "test:", len(X_test))

preprocessor_numeric = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])
preprocessor_numeric_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True))
])
preprocessor_categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5, dtype=np.float32))
])

def make_preprocessor(numeric_transformer):
    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", preprocessor_categorical, categorical_features)
    ])

tuning_cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)


Features: ['usage_count', 'log_usage_count', 'unique_days_used', 'log_unique_days_used', 'days_since_last_use', 'active_days', 'usage_frequency', 'success_rate', 'smoothed_failure_rate', 'risk_weight', 'scope_breadth', 'is_wildcard_resource', 'is_delete', 'is_permission_change', 'is_security_sensitive', 'service', 'operation_type', 'risk_level', 'action_family', 'position', 'department', 'team']
Rows -> train: 12000 validation: 4000 test: 4000


In [2]:
estimator = Pipeline([
    ("preprocessor", make_preprocessor(preprocessor_numeric)),
    ("model", __import__("sklearn.linear_model", fromlist=["LogisticRegression"]).LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=3000,
        random_state=42
    ))
])

In [3]:
param_space = {
    "model__C": loguniform(0.01, 10.0)
}

In [4]:
search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=param_space,
    n_iter=8,
    scoring="average_precision",
    cv=tuning_cv,
    random_state=42,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    verbose=1
)

search.fit(X_fit, y_fit, groups=groups_fit)

print("Best CV PR-AUC:", round(search.best_score_, 4))
print("Best params:")
print(search.best_params_)

best_estimator = search.best_estimator_
val_prob = best_estimator.predict_proba(X_val)[:, 1]
val_pr_auc = average_precision_score(y_val, val_prob)
print("Validation PR-AUC:", round(val_pr_auc, 4))

Fitting 4 folds for each of 8 candidates, totalling 32 fits


Best CV PR-AUC: 0.5531
Best params:
{'model__C': np.float64(7.114476009343421)}
Validation PR-AUC: 0.5906


In [5]:
artifact = {
    "model_name": "logistic_regression",
    "estimator": best_estimator,
    "best_cv_pr_auc": float(search.best_score_),
    "validation_pr_auc": float(val_pr_auc),
    "feature_columns": all_features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "best_params": search.best_params_
}
joblib.dump(artifact, ARTIFACT_DIR / "logistic_regression_candidate.joblib")
print("Saved:", ARTIFACT_DIR / "logistic_regression_candidate.joblib")

Saved: C:\Users\LENOVO\Downloads\IAM_Model_V2_Notebooks\artifacts\logistic_regression_candidate.joblib
